In [ ]:
# Install dependencies
!pip install -q numpy pybullet gym==0.21.0 stable-baselines3

In [ ]:
%%bash
cat > robot_env.py <<'PY'
import gym
import numpy as np
import pybullet as p
import pybullet_data
from gym import spaces
import time

class SimpleArmEnv(gym.Env):
    metadata = {'render.modes': ['human']}
    def __init__(self, urdf_path="kuka_iiwa/model.urdf", render=False):
        super().__init__()
        self.render_mode = render
        if render:
            p.connect(p.GUI)
        else:
            p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)
        self.plane = p.loadURDF("plane.urdf")
        self.robot = p.loadURDF(urdf_path, [0,0,0], useFixedBase=True)
        self.joint_indices = [i for i in range(p.getNumJoints(self.robot))
                              if p.getJointInfo(self.robot, i)[2] != p.JOINT_FIXED][:6]
        self.n = len(self.joint_indices)
        self.action_space = spaces.Box(low=-0.1, high=0.1, shape=(self.n,), dtype=np.float32)
        obs_dim = self.n + self.n + 3 + 3
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)
        self.target = np.array([0.5, 0.0, 0.5])
        self.max_steps = 200
        self.step_counter = 0
    def reset(self):
        for j in self.joint_indices:
            p.resetJointState(self.robot, j, targetValue=0.0)
        self.target = np.array([0.5, 0.0, 0.5]) + 0.05 * np.random.randn(3)
        self.step_counter = 0
        return self._get_obs()
    def _get_ee_pos(self):
        link_id = self.joint_indices[-1]
        link_state = p.getLinkState(self.robot, link_id)
        pos = np.array(link_state[0])
        return pos
    def _get_obs(self):
        q = []
        qdot = []
        for j in self.joint_indices:
            s = p.getJointState(self.robot, j)
            q.append(s[0]); qdot.append(s[1])
        ee = self._get_ee_pos()
        obs = np.concatenate([np.array(q), np.array(qdot), ee, self.target])
        return obs.astype(np.float32)
    def step(self, action):
        action = np.clip(action, self.action_space.low, self.action_space.high)
        for a, j in zip(action, self.joint_indices):
            st = p.getJointState(self.robot, j)
            newpos = st[0] + float(a)
            p.setJointMotorControl2(self.robot, j, p.POSITION_CONTROL, targetPosition=newpos, force=200)
        for _ in range(8):
            p.stepSimulation()
        obs = self._get_obs()
        ee = obs[-6:-3]
        dist = np.linalg.norm(ee - self.target)
        r_goal = -dist
        r_ctrl = -0.01 * (action**2).sum()
        done = False
        if dist < 0.05:
            r_goal += 5.0
            done = True
        self.step_counter += 1
        if self.step_counter >= self.max_steps:
            done = True
        reward = float(r_goal + r_ctrl)
        info = {"dist": float(dist)}
        return obs, reward, done, info
    def render(self, mode='human'):
        pass
    def close(self):
        try:
            p.disconnect()
        except Exception:
            pass
PY

cat > train_sb3.py <<'PY'
import gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from robot_env import SimpleArmEnv

def make_env():
    return SimpleArmEnv(urdf_path="kuka_iiwa/model.urdf", render=False)

if __name__ == "__main__":
    num_envs = 4
    env = DummyVecEnv([make_env for _ in range(num_envs)])
    env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10, gamma=0.99)
    model.learn(total_timesteps=50_000)  # Colab demo: keep small
    model.save("ppo_kuka_arm_colab")

PY

In [ ]:
# Run a quick training (Colab demo)
!python train_sb3.py